# 3. Rendimiento en NVIDIA Jetson Orin Nano
Si los notebooks 01 y 02 se ejecutaron en esta misma placa, dejar `TRANSFER = None`: los runs y las trazas locales son la referencia y no se necesita bundle. Si los modelos se entrenaron en otro equipo, indicar la ruta de su bundle privado.


In [ ]:
from pathlib import Path
import json
import platform
import sys
import pandas as pd
import os
raiz = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'shared' / 'contract.py').is_file()), None)
assert raiz is not None, f'No se encuentra la raíz del repositorio desde {Path.cwd()}'
os.chdir(raiz)
sys.path.insert(0, str(raiz))
from shared.contract import ROOT, validate_splits
from shared.reporting.edge_bundle import verify, install
assert Path.cwd().resolve() == ROOT
assert sys.platform.startswith('linux')
print(platform.platform(), platform.machine(), sys.version)
TRANSFER = None  # Para un bundle externo: Path('/ruta/privada/al/bundle')
LIGHTGBM_RUN = None  # Opcional: se comprobará contra el informe emparejado
LSTM_RUN = None      # Opcional: se comprobará contra el informe emparejado


## Selección y verificación
En modo directo se seleccionan los runs por `model_version` del informe emparejado del notebook 02. Pueden existir runs anteriores; no se borra ninguno. Si se indican rutas manuales, deben coincidir con el informe. En modo bundle se verifican e instalan archivos ignorados sin sobrescribir ninguno distinto.


In [ ]:
if TRANSFER is None:
    contract = validate_splits()
    quality = json.loads((ROOT / 'results' / 'metrics' / 'paired_test.json').read_text(encoding='utf-8'))
    assert quality['paired_test_rows'] == 31680
    assert all(quality[name]['data_manifest_id'] == contract['manifest_id'] for name in ('lightgbm', 'lstm'))
    def select_run(kind, approach, selected):
        version = quality[approach]['model_version']
        prefix = {'lightgbm_night_v1': 'lightgbm-night-context-v1-', 'lstm_causal_v1': 'lstm-causal-v1-', 'lstm_causal_v2': 'lstm-causal-v2-'}[kind]
        assert version.startswith(prefix), f'{approach}: versión de modelo inesperada: {version}'
        run_id = version[len(prefix):]
        assert run_id and Path(run_id).name == run_id, f'{approach}: ID de run inválido'
        expected = (ROOT / 'experiments' / 'runs' / kind / run_id).resolve()
        assert (expected / 'manifest.json').is_file(), f'{approach}: falta el run del informe: {expected}'
        if selected is not None:
            assert (ROOT / selected).resolve() == expected, f'{approach}: el run elegido no coincide con el informe emparejado'
        return expected
    lightgbm_run = select_run('lightgbm_night_v1', 'lightgbm', LIGHTGBM_RUN)
    lstm_run = select_run('lstm_causal_v2' if quality['lstm']['model_version'].startswith('lstm-causal-v2-') else 'lstm_causal_v1', 'lstm', LSTM_RUN)
else:
    plan = verify(TRANSFER)
    install(TRANSFER)
    assert validate_splits()['manifest_id'] == plan['data_manifest_id']
    runs = json.loads((TRANSFER / 'bundle_manifest.json').read_text(encoding='utf-8'))['runs']
    lightgbm_run, lstm_run = ROOT / runs['lightgbm'], ROOT / runs['lstm']
print(lightgbm_run, lstm_run, sep='\n')


## Paridad de inferencia antes del cronómetro
Se cargan primero las trazas de referencia en memoria; luego se vuelve a inferir y se comprueba que las predicciones no cambian. En modo directo, las referencias proceden del notebook 01 y del informe validado por el notebook 02.


In [ ]:
from approaches.lightgbm.night_context import export_test as export_lightgbm
from approaches.lstm.adapter import export_test as export_lstm
reference_root = ROOT / ('results/traces' if TRANSFER is None else 'experiments/runs/edge_reference')
expected = {name: pd.read_parquet(reference_root / f'{name}_test.parquet') for name in ('lightgbm_night', 'lstm')}
assert len(expected['lightgbm_night']) == len(expected['lstm']) == 31680
assert set(expected['lightgbm_night'].model_version) == {f'lightgbm-night-context-v1-{lightgbm_run.name}'}
assert set(expected['lstm'].model_version) == {f'lstm-causal-{lstm_run.parent.name[-2:]}-{lstm_run.name}'}
if TRANSFER is None:
    assert quality['lightgbm']['model_version'] == expected['lightgbm_night'].model_version.iloc[0]
    assert quality['lstm']['model_version'] == expected['lstm'].model_version.iloc[0]
for name, produced in [('lightgbm_night', export_lightgbm(lightgbm_run)[0]), ('lstm', export_lstm(lstm_run)[0])]:
    actual = pd.read_parquet(produced)
    columns = ['timestamp', 'ground_truth_type', 'binary_prediction', 'stage_2_prediction', 'final_prediction']
    assert actual[columns].equals(expected[name][columns]), f'{name}: la inferencia cambió'
    print(name, len(actual), 'objetivos idénticos')


## Medición
Ambos benchmarks registran plataforma, versiones, calentamiento, repeticiones, unidad y alcance. LightGBM incluye extracción causal y reglas; LSTM usa ventanas ya preparadas. Mostrar ambos alcances y no interpretar sus latencias como equivalentes.


In [ ]:
from approaches.lightgbm.night_benchmark import run_benchmark as benchmark_lightgbm
from approaches.lstm.benchmark import run_benchmark as benchmark_lstm
lightgbm_report = benchmark_lightgbm(lightgbm_run, rows=2000, repeats=5)
lstm_report = benchmark_lstm(lstm_run, rows=2000, repeats=5)
for path in (lightgbm_report, lstm_report):
    report = json.loads(path.read_text(encoding='utf-8'))
    print(path, report['prediction_unit'], report['timing_scope'], report['median_latency_ms'], report['p95_latency_ms'])


Los informes quedan en `results/reports/`, ignorados por Git. Registrar además JetPack, modo de potencia, frecuencia y temperatura durante cada medición si se usan para interpretar diferencias; no aparecen todavía en el esquema común.
